# Space-Grade Fault Detector — LibreLane, staged flow
Run from `/foss/designs/Space-Grade-Mechanical-Fault-Detector/`.
Three separate config files, run one at a time: `config_lint.yaml` → `config_synth.yaml` → `config_openroad.yaml`.
Each stage is a standalone LibreLane run (re-reads RTL) — this is a staged review process, not a single continuous flow with shared state.

## 1. Environment check + confirm top.v ports

In [1]:
import os, yaml
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("CWD:", os.getcwd())
with open("../../rtl/top.v") as f:
    print(f.read()[:2000])  # read the module port declaration — fill CLOCK_PORT below with the real name

PDK_ROOT: /foss/pdks
CWD: /foss/designs/Space-Grade-Mechanical-Fault-Detector/librelane/test2
//============================================================================
// top.v -- Space-Grade Vibration Pattern Anomaly Detector
// Integrates: spi_apb_interface (owns spi_master), axis_sequencer,
// goertzel_core, magnitude_compute, fault_flagger, tmr_reg_bank, apb
//============================================================================
`timescale 1ns/1ps
`default_nettype none

`include "spi_apb_interface.v"
`include "axis_sequencer.v"
`include "goertzel_core.v"
`include "magnitude_compute.v"
`include "fault_flagger.v"
`include "tmr_reg_bank.v"

module top (
    input  wire        clk,
    input  wire        sys_rst_n,

    // IIS3DWB sensor SPI pins
    input  wire        c_miso,
    output wire        c_csn,
    output wire        c_sclk,
    output wire        c_mosi,

    // DRDY interrupt from sensor
    input  wire        sensor_drdy,

    // tmr_forward_en: 0=Option A (l

## 2. Stage 1 — Lint only (`config_lint.yaml`)
Verilator.Lint plus its checker steps. Catches RTL errors before anything else runs.

In [2]:
from librelane.flows import SequentialFlow
from librelane.steps import Verilator, Checker

class LintFlow(SequentialFlow):
    Steps = [
        Verilator.Lint,
        Checker.LintTimingConstructs,
        Checker.LintErrors,
        Checker.LintWarnings,
    ]

with open("config_lint.yaml") as f:
    lint_config = yaml.safe_load(f)

lint_flow = LintFlow(lint_config, design_dir="../..")
lint_state, lint_steps = lint_flow.start()

[06:30:23] INFO     Starting a new run of the 'LintFlow' flow with the tag 'RUN_2026-07-22_06-30-23'.   ]8;id=16475738;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py\flow.py]8;;\:]8;id=16475739;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py#654\654]8;;\

Output()

[06:30:23] INFO     Starting…                                                                     ]8;id=16475746;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/sequential.py\sequential.py]8;;\:]8;id=16475747;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/sequential.py#339\339]8;;\

───────────────────────────────────────────────── Verilator Lint ──────────────────────────────────────────────────

[06:30:23] VERBOSE  Running 'Verilator.Lint' at '../../runs/RUN_2026-07-22_06-30-23/1-verilator-lint'… ]8;id=16475754;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py\step.py]8;;\:]8;id=16475755;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py#1138\1138]8;;\

...

[06:30:25] INFO     Check for Lint Timing Errors clear.                                              ]8;id=16475775;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/checker.py\checker.py]8;;\:]8;id=16475776;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/checker.py#412\412]8;;\

─────────────────────────────────────────────── Lint Errors Checker ───────────────────────────────────────────────

[06:30:25] VERBOSE  Running 'Checker.LintErrors' at                                                    ]8;id=16475781;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py\step.py]8;;\:]8;id=16475782;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py#1138\1138]8;;\
                    '../../runs/RUN_2026-07-22_06-30-23/3-checker-linterrors'…                                     

[06:30:25] ERROR    21 Lint errors found.                                                            ]8;id=16475788;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/checker.py\checker.py]8;;\:]8;id=16475789;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/checker.py#127\127]8;;\

FlowError: 21 Lint errors found.

## 3. Stage 2 — Synthesis only (`config_synth.yaml`)
Yosys.Synthesis plus its checkers. Fill in CLOCK_PORT in `config_synth.yaml` from cell 1's output before running this.

In [3]:
from librelane.steps import Yosys

class SynthFlow(SequentialFlow):
    Steps = [
        Yosys.Synthesis,
        Checker.YosysUnmappedCells,
        Checker.YosysSynthChecks,
        Checker.NetlistAssignStatements,
    ]

with open("config_synth.yaml") as f:
    synth_config = yaml.safe_load(f)

synth_flow = SynthFlow(synth_config, design_dir="../..")
synth_state, synth_steps = synth_flow.start()

[06:30:30] INFO     Starting a new run of the 'SynthFlow' flow with the tag 'RUN_2026-07-22_06-30-30'.  ]8;id=16475794;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py\flow.py]8;;\:]8;id=16475795;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/flow.py#654\654]8;;\

Output()

[06:30:30] INFO     Starting…                                                                     ]8;id=16475800;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/sequential.py\sequential.py]8;;\:]8;id=16475801;file:///usr/local/lib/python3.12/dist-packages/librelane/flows/sequential.py#339\339]8;;\

──────────────────────────────────────────────────── Synthesis ────────────────────────────────────────────────────

[06:30:30] VERBOSE  Running 'Yosys.Synthesis' at                                                       ]8;id=16475806;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py\step.py]8;;\:]8;id=16475807;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py#1138\1138]8;;\
                    '../../runs/RUN_2026-07-22_06-30-30/1-yosys-synthesis'…                                        

...

/foss/designs/Space-Grade-Mechanical-Fault-Detector/rtl/apb.v:13: ERROR: Re-definition of module `$abstract\apb'!

[06:30:31] ERROR    Subprocess had a non-zero exit.                                                    ]8;id=16475819;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py\step.py]8;;\:]8;id=16475820;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py#1383\1383]8;;\

[06:30:31] ERROR    Last 10 line(s):                                                                   ]8;id=16475826;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py\step.py]8;;\:]8;id=16475827;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py#1388\1388]8;;\
                    Successfully finished Verilog frontend.                                                        
                                                                                                                   
                    8. Executing Verilog-2005 frontend:                                                            
                    /foss/designs/Space-Grade-Mechanical-Fault-Detector/rtl/magnitude_compute.v                    
                    Parsing SystemVerilog input from                                                               
                    `/foss/designs/Space-Grade-Mechanical-Fault-Detector/rtl/magnitude_compute.v' to               
                    AST representation.                                                                            
                    Storing AST representation for module `$abstract\magnitude_compute'.                           
                    Successfully finished Verilog frontend.                                                        
                                                                                                                   
                    9. Executing Verilog-2005 frontend:                                                            
                    /foss/designs/Space-Grade-Mechanical-Fault-Detector/rtl/spi_apb_interface.v                    
                    Parsing SystemVerilog input from                                                               
                    `/foss/designs/Space-Grade-Mechanical-Fault-Detector/rtl/spi_apb_interface.v' to               
                    AST representation.                                                                            
                    /foss/designs/Space-Grade-Mechanical-Fault-Detector/rtl/apb.v:13: ERROR:                       
                    Re-definition of module `$abstract\apb'!                                                       
                                                                                                                   

[06:30:31] ERROR    Full log file:                                                                     ]8;id=16475833;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py\step.py]8;;\:]8;id=16475834;file:///usr/local/lib/python3.12/dist-packages/librelane/steps/step.py#1391\1391]8;;\
                    '../../runs/RUN_2026-07-22_06-30-30/1-yosys-synthesis/yosys-synthesis.log'                     

FlowError: Synthesis: subprocess (1, ['yosys', '-y', '/usr/local/lib/python3.12/dist-packages/librelane/scripts/pyosys/synthesize.py', '--', '--config-in', '/foss/designs/Space-Grade-Mechanical-Fault-Detector/runs/RUN_2026-07-22_06-30-30/1-yosys-synthesis/config.json', '--extra-in', '/foss/designs/Space-Grade-Mechanical-Fault-Detector/runs/RUN_2026-07-22_06-30-30/1-yosys-synthesis/extra.json', '--output', '/foss/designs/Space-Grade-Mechanical-Fault-Detector/runs/RUN_2026-07-22_06-30-30/1-yosys-synthesis/top.nl.v']) failed

## 4. Verify TMR flop count before proceeding
Check the Yosys stats report under this run's `reports/synthesis/` for `tmr_reg_bank` —
confirm the flop count is still 3x, not deduplicated. Don't proceed to stage 3 until this is confirmed.

In [ ]:
print(synth_flow.dir)  # run directory for this stage — inspect reports/synthesis/*.stat.rpt inside it

## 5. Stage 3 — Complete OpenROAD flow (`config_openroad.yaml`)
Full RTL-to-GDSII via the built-in `Classic` flow. Uses `constraints.sdc` via `FALLBACK_SDC` for timing closure.
`RUN_LINTER: false` in this config skips re-linting (already done in stage 1); synthesis still runs internally — Classic has no built-in resume-from-netlist without `--last-run`, so this is expected, not wasted work.

In [ ]:
from librelane.flows import Flow

Classic = Flow.factory.get("Classic")

with open("config_openroad.yaml") as f:
    openroad_config = yaml.safe_load(f)

pnr_flow = Classic(openroad_config, design_dir=".")
pnr_state, pnr_steps = pnr_flow.start()

## 6. Post-flow checks
Inspect `reports/signoff/` in this run's directory for STA (setup/hold slack), DRC, and LVS results
before treating `top` as a hardened macro ready for the pad ring / Chip flow stage.

In [ ]:
print(pnr_flow.dir)